# Ticket scarcity and appropriate reliance on AI — V2
Qianshuo Wang | PS1 revision

Synthetic classroom model. No human participants and no real AI model. Uses Python standard library only. The baseline model preserves the original scarcity mapping, while V2 adds functional-form sensitivity checks using weaker and stronger scarcity mappings.

The original game uses deterministic scenario hashes and retrospective correctness. Here we explicitly assume a Bernoulli sell-out state and a symmetric noisy state signal with conditional accuracy r. This stochastic extension is not an estimate of real ticket-market calibration.




## Author verification

On September 20, 2026, I independently ran all cells in this v2 notebook
in Google Colab and reviewed the generated outputs. The baseline
25-condition grid and the 75-condition functional-form robustness analysis
executed successfully. I also verified that the posterior-policy dominance
and robustness checks passed.

In [1]:
"""Synthetic ticket decision model. No human data or actual AI predictions."""
import math, random, csv, json
from pathlib import Path

def risk(Q, N, k=8):
    """Assumed sell-out probability. k=8 reproduces the original baseline mapping."""
    if not all(math.isfinite(x) for x in (Q, N, k)) or Q < 1 or N < 0 or k <= 0:
        raise ValueError('Q and k must be positive; N must be nonnegative')
    return min(.98, N/(N+k*Q+1))

def posterior(p, r, advice):
    if not (0 <= p <= 1 and 0 <= r <= 1) or advice not in ('buy', 'wait'):
        raise ValueError('invalid probability or advice')
    # Symmetric noisy signal of sell-out state; not an empirical accuracy claim.
    a, b = (r, 1-r) if advice == 'buy' else (1-r, r)
    denom = p*a + (1-p)*b
    return None if denom == 0 else p*a/denom

def values(P, V, p):
    if not (math.isfinite(P) and math.isfinite(V) and 0 < P < V and 0 <= p <= 1):
        raise ValueError('analysis restricts to 0 < P < V and p in [0,1]')
    return V-P, (1-p)*(V-.9*P)

def action(P, V, p):
    b, w = values(P, V, p)
    return 'buy' if b >= w else 'wait'

def expected(P, V, p, r, policy):
    total = 0.
    for sold in (True, False):
        for advice in ('buy', 'wait'):
            prob = (p if sold else 1-p) * (r if ((advice == 'buy') == sold) else 1-r)
            if prob == 0:
                continue
            post = posterior(p, r, advice)
            choice = advice if policy == 'follow' else action(
                P, V, p if policy == 'ignore' else post
            )
            reward = V-P if choice == 'buy' else (0 if sold else V-.9*P)
            total += prob * reward
    return total

def run(outdir):
    """Original 25-condition baseline grid using k=8."""
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    rows = []

    for N in (10, 50, 100, 500, 900):
        for r in (.3, .5, .6, .8, 1.):
            p = risk(40, N, k=8)
            row = {'N': N, 'Q': 40, 'P': 150, 'V': 220,
                   'reliability': r, 'risk': p}
            for policy in ('ignore', 'follow', 'bayes'):
                row[policy] = expected(150, 220, p, r, policy)
            row['follow_regret'] = row['bayes'] - row['follow']
            rows.append(row)

    with (outdir/'policy_comparison.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)

    p = risk(40, 500, k=8)
    r = .8
    rng = random.Random(206)
    totals = {k: 0. for k in ('ignore', 'follow', 'bayes')}
    trials = 100000

    for _ in range(trials):
        sold = rng.random() < p
        correct = rng.random() < r
        advice = 'buy' if sold == correct else 'wait'
        for policy in totals:
            choice = advice if policy == 'follow' else action(
                150, 220, p if policy == 'ignore' else posterior(p, r, advice)
            )
            totals[policy] += 70 if choice == 'buy' else (0 if sold else 85)

    result = {
        'evidence': 'synthetic exact enumeration and Monte Carlo; not human behavior',
        'seed': 206,
        'trials': trials,
        'risk': p,
        'no_advice_threshold': 15/85,
        'posterior_buy': posterior(p, r, 'buy'),
        'posterior_wait': posterior(p, r, 'wait'),
        'exact': {k: expected(150, 220, p, r, k) for k in totals},
        'monte_carlo': {k: v/trials for k, v in totals.items()},
        'grid_cases': len(rows)
    }

    (outdir/'summary.json').write_text(json.dumps(result, indent=2) + '\n')
    return result

def robustness(outdir):
    """V2 functional-form sensitivity: k in {4,8,12}, 25 conditions each."""
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    rows = []

    for k in (4, 8, 12):
        for N in (10, 50, 100, 500, 900):
            for r in (.3, .5, .6, .8, 1.):
                p = risk(40, N, k)
                ignore = expected(150, 220, p, r, 'ignore')
                follow = expected(150, 220, p, r, 'follow')
                bayes = expected(150, 220, p, r, 'bayes')

                rows.append({
                    'k': k,
                    'N': N,
                    'r': r,
                    'risk': p,
                    'ignore': ignore,
                    'follow': follow,
                    'bayes': bayes,
                    'follow_regret': bayes - follow,
                    'ignore_regret': bayes - ignore
                })

    with (outdir/'robustness.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0])
        writer.writeheader()
        writer.writerows(rows)

    summary = {}
    for k in (4, 8, 12):
        subset = [x for x in rows if x['k'] == k]
        summary[k] = {
            'cases': len(subset),
            'bayes_weakly_dominates_follow':
                all(x['bayes'] + 1e-10 >= x['follow'] for x in subset),
            'bayes_weakly_dominates_ignore':
                all(x['bayes'] + 1e-10 >= x['ignore'] for x in subset),
            'mean_follow_regret':
                sum(x['follow_regret'] for x in subset) / len(subset),
            'max_follow_regret':
                max(x['follow_regret'] for x in subset)
        }

    (outdir/'robustness_summary.json').write_text(
        json.dumps(summary, indent=2) + '\n'
    )
    return rows, summary


## Exact enumeration and V2 robustness extension
Baseline: follow every recommendation. Main comparator: choose the action with the larger Bayesian posterior expected payoff. Ignore-advice is an additional comparator. Each condition enumerates two states and two recommendations.

The original scarcity mapping uses $k=8$ in $p_k=\min\{.98, N/(N+kQ+1)\}$. V2 adds $k=4$ and $k=12$ as stronger/weaker scarcity mappings. These are sensitivity specifications, not empirically estimated market probabilities.


In [2]:
result = run('ps1_outputs')
print('Baseline result:')
print(json.dumps(result, indent=2))

robust_rows, robust_summary = robustness('ps1_outputs')
print('\nRobustness check:')
print(json.dumps(robust_summary, indent=2))
print('\nTotal robustness conditions:', len(robust_rows))


Baseline result:
{
  "evidence": "synthetic exact enumeration and Monte Carlo; not human behavior",
  "seed": 206,
  "trials": 100000,
  "risk": 0.6090133982947625,
  "no_advice_threshold": 0.17647058823529413,
  "posterior_buy": 0.8616975441619992,
  "posterior_wait": 0.2802690582959641,
  "exact": {
    "ignore": 70.0,
    "follow": 66.16565164433618,
    "bayes": 70.0
  },
  "monte_carlo": {
    "ignore": 70.0,
    "follow": 66.202,
    "bayes": 70.0
  },
  "grid_cases": 25
}

Robustness check:
{
  "4": {
    "cases": 25,
    "bayes_weakly_dominates_follow": true,
    "bayes_weakly_dominates_ignore": true,
    "mean_follow_regret": 9.990111515686957,
    "max_follow_regret": 40.88171536286521
  },
  "8": {
    "cases": 25,
    "bayes_weakly_dominates_follow": true,
    "bayes_weakly_dominates_ignore": true,
    "mean_follow_regret": 7.764869257235967,
    "max_follow_regret": 34.934889434889435
  },
  "12": {
    "cases": 25,
    "bayes_weakly_dominates_follow": true,
    "bayes_wea

## Independent checks
The original threshold, posterior-policy dominance, and Monte Carlo checks are retained. V2 additionally verifies 75 robustness conditions across three scarcity mappings. The Bayesian posterior-optimal policy must weakly dominate the fixed follow and ignore policies under the maintained model.


In [3]:
assert abs(values(150,220,15/85)[0]-values(150,220,15/85)[1]) < 1e-10

for p in (0,.01,.1,.61,.98,1):
    for r in (0,.3,.5,.8,1):
        for policy in ('follow','ignore'):
            assert expected(150,220,p,r,'bayes') + 1e-10 >= expected(150,220,p,r,policy)

assert abs(result['monte_carlo']['follow'] - result['exact']['follow']) < 0.5
assert len(robust_rows) == 75

for row in robust_rows:
    assert row['bayes'] + 1e-10 >= row['follow']
    assert row['bayes'] + 1e-10 >= row['ignore']

print('Threshold, posterior-policy dominance, Monte Carlo, and robustness checks passed.')


Threshold, posterior-policy dominance, Monte Carlo, and robustness checks passed.


## Interpretation and next human test
At the default market inputs, even Wait advice at r=0.8 leaves posterior sell-out probability above the buy threshold, so always following advice loses expected utility in that condition. This does not show that people overrely on AI.

V2 adds a functional-form sensitivity analysis because the sell-out probability is assumed rather than empirically estimated. The robustness check asks whether the policy comparison survives weaker and stronger scarcity mappings.

For future human validation, observed advice-following should not be interpreted as trust by itself. Greater reliance on more accurate advice may reflect rational updating. A stronger design would elicit prior/posterior sell-out beliefs and vary perceived source reputation while holding objective advice quality constant.

Sources: Li, Lu & Yin (2023), https://doi.org/10.1609/aaai.v37i5.25748 ; Simon (1955), https://doi.org/10.2307/1884852. Neither paper supplies the numerical risk formula or signal calibration; those remain classroom assumptions.

AI disclosure: AI tools assisted with model formalization, coding, debugging, and consistency checks after the author's initial reasoning. The author remains responsible for reviewing assumptions, rerunning the notebook, and verifying final outputs.
